# LongFlow — P2 MeanFlow run (mf_640 on capture v3)

Runtime: **A100**, ~2–2.5 h. Pre-registration: NOTES "P2 MEANFLOW
PRE-REGISTRATION" (2026-08-21). Core verified locally (JVP vs float64
finite differences 1.9e-05; adaptive weighting proven load-bearing;
synthetic NFE-2 relRMSE 0.08). 108/108 tests green.

The head learns the AVERAGE velocity u(x_t, t→r); NFE 1–2 is its home
regime. Target: the frame-boundary sampling scatter ("tiny voice frames
mismatched ever so slightly"); acceptance test: make 2:25–2:50 the whole
render. New instrument `interframe_scatter` reports student-vs-teacher
adjacent-frame jump stats on every eval.

| cell | what |
|---|---|
| 1 | cold start (v3 cache local) |
| 2 | pool + identical held-out split |
| 3 | **GATE: mf head 5K + clips → Drive → LISTEN** (constraint 6; if the loss explodes here, stop — retry at lr 1e-4) |
| 4 | train mf_640: 40K steps, ckpts every 5K |
| 5 | held-out renders (NFE 2 per ckpt; NFE 1/2/8-heun at 40K) + SCATTER |
| 6 | 30 s-chunked closed loop: qf_mf_nfe2 / qf_mf_nfe1 |
| 7 | bundle → `quality_eval3.zip` (same scorer notebook as before) |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "MeanFlow v1.0 (2026-08-21): mf_640 on capture v3"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import FlowHeadPatch, _CFGField
from src.flow_head.meanflow import (
    RTEqualField, interframe_scatter, meanflow_loss, mf_cfg_sample, mf_sample,
)
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import load_checkpoint, pairs_from_files, train
print("meanflow import OK — repo has the P2 commit")

CACHE_V3_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v3"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/mf"
DRIVE_OUT = "/content/drive/MyDrive/longflow_quality"
EVAL_DIR = "/content/mf_eval"
for d in (OUT, EVAL_DIR, f"{DRIVE_OUT}/mfgate"):
    os.makedirs(d, exist_ok=True)

LOCAL_CACHE = "/content/cache_v3"
n_drive = len(glob.glob(f"{CACHE_V3_DRIVE}/*.pt"))
if not os.path.exists(LOCAL_CACHE) or len(glob.glob(f"{LOCAL_CACHE}/*.pt")) < n_drive:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print(f"bulk-copying {n_drive} v3 files (~15 GB)...", flush=True)
    !cp {CACHE_V3_DRIVE}/*.pt {LOCAL_CACHE}/
print(f"{len(glob.glob(f'{LOCAL_CACHE}/*.pt'))} cache files local")

if os.path.exists(f"{DRIVE_OUT}/mf_report.json"):
    with open(f"{DRIVE_OUT}/mf_report.json") as f:
        report = json.load(f)
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": [], "scatter": {}}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/mf_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt failed: {repr(e)[:120]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

def mf_latents(head, mean_t, std_t, utt, nfe, seed=0):
    g = torch.Generator(device="cuda").manual_seed(seed)
    z = mf_cfg_sample(head, utt.hidden.float().cuda(), utt.neg_hidden.float().cuda(),
                      head.cfg.d_latent, cfg_scale=1.3, nfe=nfe, generator=g)
    return z * std_t.cuda() + mean_t.cuda()

def heun_rt_latents(head, mean_t, std_t, utt, seed=0):
    field = _CFGField(RTEqualField(head), utt.neg_hidden.float().cuda(), 1.3)
    g = torch.Generator(device="cuda").manual_seed(seed)
    z = heun_sample(field, utt.hidden.float().cuda(), head.cfg.d_latent,
                    nfe=8, sway=0.0, generator=g)
    return z * std_t.cuda() + mean_t.cuda()

print("READY")


In [ ]:
# ===== Pool + identical held-out split (cond-only load) =====
HELD_OUT_PER_BIN = 5
all_files = sorted(glob.glob(f"{LOCAL_CACHE}/*.pt"))

def fname_bin(path):
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in all_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print(f"held-out {len(held_out_files)} / train {len(train_files)} (identical split to the ladder)")

data = pairs_from_files(train_files, dual_stream=False)
print(f"pool: {data.hidden.shape[0]} frames")

MF_LOSS = lambda h, x, c: meanflow_loss(h, x, c, p_equal=0.75)


## 3. GATE — hard constraint 6 (LISTEN before cell 4)

5K steps of the mf head, 2 clips at NFE 2 to
`Drive/longflow_quality/mfgate/`. Expected: rough 5K texture but SPEECH.
Also watch the printed loss — with the adaptive weighting it should sit
O(1) and drift down; explosion = STOP (retry at lr 1e-4, then report).


In [ ]:
if glob.glob(f"{DRIVE_OUT}/mfgate/*_gate.wav"):
    print("gate clips already on Drive — skip to listening / cell 4")
else:
    gate_head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent,
                                        meanflow=True))
    print(f"gate head: {gate_head.param_count()/1e6:.2f}M params")
    out = train(gate_head, data, steps=5000, batch_size=1024, lr=2e-4,
                ema_decay=0.999, device="cuda", log_every=500, loss_fn=MF_LOSS)
    out["ema"].copy_to(gate_head)
    gate_head.eval()
    for fpath in (held_out_by_bin[150][0], held_out_by_bin[1200][0]):
        utt = load_utterance(fpath)
        sf.write(f"{DRIVE_OUT}/mfgate/{utt.utt_id}_teacher.wav",
                 decode_latents(utt.latent.float()), 24000)
        z = mf_latents(gate_head, data.mean, data.std, utt, nfe=2)
        sf.write(f"{DRIVE_OUT}/mfgate/{utt.utt_id}_gate.wav", decode_latents(z), 24000)
        print(f"gate clips for {utt.utt_id} on Drive", flush=True)
    del gate_head
    torch.cuda.empty_cache()
    print("\nLISTEN NOW (Drive/longflow_quality/mfgate/) — run cell 4 only on a PASS.")


In [ ]:
# ===== Train mf_640: 40K steps, ckpts every 5K =====
TAG = "mf_640"
final = f"{CKPT_DIR}/{TAG}_step40000.pt"
if os.path.exists(final):
    print(f"{TAG}: final checkpoint already on Drive — skipping")
else:
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent,
                                   meanflow=True))
    print(f"{TAG}: {head.param_count()/1e6:.2f}M params")
    t0 = time.time()
    train(head, data, steps=40000, batch_size=1024, lr=2e-4, lr_final=2e-5,
          ema_decay=0.9999, device="cuda", log_every=2000, loss_fn=MF_LOSS,
          checkpoint_every=5000,
          checkpoint_path_fn=lambda s: f"{CKPT_DIR}/{TAG}_step{s}.pt")
    print(f"done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


In [ ]:
# ===== Held-out renders + THE SCATTER INSTRUMENT =====
SUBSET_PER_BIN = 2
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}

for fpath in held_out_files:
    utt = load_utterance(fpath)
    tname = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{tname}"):
        sf.write(f"{EVAL_DIR}/{tname}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": tname, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
subset_files = [fs[:SUBSET_PER_BIN] for fs in held_out_by_bin.values()]
subset_files = [f for fs in subset_files for f in fs]

ARMS_FINAL = {"mf_nfe1": 1, "mf_nfe2": 2, "mf_nfe8h": 8}

for step in (5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000):
    p = f"{CKPT_DIR}/mf_640_step{step}.pt"
    if not os.path.exists(p):
        continue
    head, mean_t, std_t = load_checkpoint(p)
    head = head.to("cuda").eval()
    arms = ARMS_FINAL if step == 40000 else {"mf_nfe2": 2}
    for arm, nfe in arms.items():
        key = f"{step}:{arm}"
        if key in manifest["checkpoints"]:
            continue
        eval_files = held_out_files if step == 40000 else subset_files
        entries = []
        for fpath in eval_files:
            utt = load_utterance(fpath)
            name = f"{utt.utt_id}_{arm}_step{step}.wav"
            if not os.path.exists(f"{EVAL_DIR}/{name}"):
                if arm == "mf_nfe8h":
                    z = heun_rt_latents(head, mean_t, std_t, utt, seed=0)
                else:
                    z = mf_latents(head, mean_t, std_t, utt, nfe=nfe, seed=0)
                sf.write(f"{EVAL_DIR}/{name}", decode_latents(z), 24000)
                if step == 40000:  # the scatter instrument, student vs teacher
                    s_st = interframe_scatter(z.cpu())
                    s_te = interframe_scatter(utt.latent.float())
                    report["scatter"].setdefault(arm, {})[utt.utt_id] = {
                        "student": s_st, "teacher": s_te,
                        "excess": round(s_st["scatter_median"] / max(s_te["scatter_median"], 1e-6), 3),
                        "excess_jerk": round(s_st["jerk_median"] / max(s_te["jerk_median"], 1e-6), 3),
                    }
            entries.append({"utt_id": utt.utt_id, "audio": name,
                            "teacher_audio": manifest["teacher"][utt.utt_id]["audio"],
                            "text": utt.text, "target_words": fname_bin(fpath), "arm": arm})
        manifest["checkpoints"][key] = entries
        print(f"{key}: {len(entries)} renders", flush=True)
    del head
    torch.cuda.empty_cache()

with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
for arm, d in report.get("scatter", {}).items():
    ex = [v["excess_jerk"] for v in d.values()]
    print(f"SCATTER {arm}: median excess-jerk vs teacher = {sorted(ex)[len(ex)//2]:.2f} "
          f"(bar: <= 1.5; CFM incumbent unmeasured — first reading)")
with open(f"{DRIVE_OUT}/mf_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("held-out eval + scatter done")


In [ ]:
# ===== 30s-chunked closed loop: NFE 2 and NFE 1 =====
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        time.sleep(wait)
    raise RuntimeError(f"empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript_turns(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return turns

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
TURNS = turnscript_turns(ABL_WORDS)
CHUNKS, cur, cw = [], [], 0
for t in TURNS:
    cur.append(t); cw += len(t.split())
    if cw >= 80:
        CHUNKS.append("\n".join(cur) + "\n"); cur, cw = [], 0
if cur:
    CHUNKS.append("\n".join(cur) + "\n")
report["cl_script"] = "\n".join(TURNS) + "\n"
print(f"{len(CHUNKS)} chunks")

def crossfade_stitch(wavs, sr=24000, fade_s=0.25):
    n = int(sr * fade_s)
    out = wavs[0]
    for wv in wavs[1:]:
        if len(out) < n or len(wv) < n:
            out = np.concatenate([out, wv])
            continue
        fade = np.linspace(0, 1, n, dtype=np.float32)
        out[-n:] = out[-n:] * (1 - fade) + wv[:n] * fade
        out = np.concatenate([out, wv[n:]])
    return out

class MFPatch(FlowHeadPatch):
    """Closed-loop patch sampling with the MeanFlow few-step sampler
    (per-step CFG combination, mirrors the teacher's stream arithmetic)."""

    def __init__(self, *args, mf_nfe=2, **kwargs):
        super().__init__(*args, **kwargs)
        if not self.head.cfg.meanflow:
            raise ValueError("MFPatch requires a meanflow head")
        self.mf_nfe = mf_nfe

    def __enter__(self):
        self._was_instance_attr = "sample_speech_tokens" in vars(self.model)
        self._orig = getattr(self.model, "sample_speech_tokens", None)
        patch = self

        def flow_sample_mf(condition, neg_condition=None, cfg_scale=None):
            t0 = time.time()
            patch.calls += 1
            s = float(cfg_scale) if cfg_scale is not None else 1.3
            if neg_condition is not None and s != 1.0:
                z = mf_cfg_sample(patch.head, condition.float(), neg_condition.float(),
                                  patch.head.cfg.d_latent, cfg_scale=s, nfe=patch.mf_nfe)
            else:
                z = mf_sample(patch.head, condition.float(),
                              patch.head.cfg.d_latent, nfe=patch.mf_nfe)
            z = z * patch.std + patch.mean
            patch.time_s += time.time() - t0
            patch.latents.append(z.detach().float().cpu())
            return z.to(condition.dtype)

        self.model.sample_speech_tokens = flow_sample_mf
        return self

head_mf, mean_mf, std_mf = load_checkpoint(f"{CKPT_DIR}/mf_640_step40000.pt")
head_mf = head_mf.to("cuda").eval()

for tag, nfe in (("qf_mf_nfe2", 2), ("qf_mf_nfe1", 1)):
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    chunk_wavs, all_z = [], []
    for ci, chunk_text in enumerate(CHUNKS):
        torch.manual_seed(ci)
        with MFPatch(model, head_mf, mean_mf, std_mf, mf_nfe=nfe) as patch, \
             torch.inference_mode():
            gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                 tokenizer=processor.tokenizer,
                                 cfg_scale=1.3, max_new_tokens=600)
        chunk_wavs.append(gen.speech_outputs[0].detach().float().cpu().numpy().squeeze())
        if patch.latents:
            all_z.append(torch.cat(patch.latents))
        print(f"  {tag} chunk {ci+1}/{len(CHUNKS)}  head {patch.time_s/max(patch.calls,1)*1000:.1f} ms/frame", flush=True)
    if all_z:
        sc = interframe_scatter(torch.cat(all_z))
        report["scatter"].setdefault("closed_loop", {})[tag] = sc
    save_wav(tag, crossfade_stitch(chunk_wavs), {"nfe": nfe, "chunks": len(CHUNKS)})
print("closed-loop renders done — LISTEN vs 41_NEW_BEST (qf_640b_plain)")


In [ ]:
# ===== Bundle -> Drive root (same scorer notebook; newest zip wins) =====
import zipfile
with open(f"{EVAL_DIR}/quality_report.json", "w") as f:
    json.dump(report, f, indent=2)
with open(f"{DRIVE_OUT}/mf_report.json", "w") as f:
    json.dump(report, f, indent=2)
ZIP = "/content/drive/MyDrive/quality_eval3.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_quality_gpu_colab.ipynb next")
